# Case TechShop - E-commerce Analytics

Notebook principal do case. Cada questão segue o padrão `[MD explicação] -> [CODE] -> [MD análise]`, exceto Q6 (apenas markdown).

# Setup

Imports centralizados, carga do dataset e configuração de caminhos.

In [1]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = Path('../data/raw/ecommerce_vendas.csv')
INTERIM_DIR = Path('../data/interim')
SQL_DIR = Path('../sql')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('Bibliotecas carregadas!')

Bibliotecas carregadas!


# Bloco 2: Q1 — Diagnóstico de Qualidade

Mapeamento de problemas de qualidade do dataset antes de qualquer tratamento.

Checklist coberto: shape e tipos, nulos por coluna, missing disfarçado, duplicatas, domínios categóricos, resumo estatístico numérico, outliers e inconsistências cruzadas.

In [2]:
df = pd.read_csv(RAW_PATH)

# --- 1. Shape e tipos: verificar se o dataset tem as 11 colunas esperadas e se os tipos estão corretos ---
print("=== Shape ===")
print(df.shape)
print("\n=== Tipos ===")
print(df.dtypes)
print("\n=== Amostra ===")
display(df.head())

# --- 2. Nulos: identificar colunas com ausências que impactam volume, receita e análise geográfica ---
print("\n=== Nulos absolutos e porcentagem ===")
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)
display(pd.DataFrame({'nulos': nulos, 'percentual_%': percentual_nulos})[nulos > 0])

# --- 3. Missing disfarçado: capturar strings vazias que o pandas não converte automaticamente em NaN ---
print("\n=== Missing disfarçado ===")
colunas_texto = df.select_dtypes(include='object').columns
strings_vazias = [(col, (df[col].fillna('').str.strip() == '').sum())
                  for col in colunas_texto
                  if (df[col].fillna('').str.strip() == '').sum() > 0]
if strings_vazias:
    print('\n'.join(f'  {col}: {n}' for col, n in strings_vazias))
else:
    print('Nenhum')

# --- 4. Duplicatas: confirmar se pedido_id é chave primária confiável antes de qualquer agregação ---
print("\n=== Duplicatas em pedido_id ===")
linhas_duplicadas = df[df.duplicated('pedido_id', keep=False)]
print(f"Linhas duplicadas: {len(linhas_duplicadas)}  |  pedido_ids afetados: {linhas_duplicadas['pedido_id'].nunique()}")
display(linhas_duplicadas.sort_values('pedido_id'))

# --- 5. Domínios: detectar valores fora do padrão em status, categoria e uf que fragmentariam agrupamentos ---
print("\n=== Distribuição: status ===")
display(df['status'].value_counts(dropna=False))

print("\n=== Distribuição: categoria ===")
display(df['categoria'].value_counts(dropna=False))

print("\n=== UFs presentes ===")
UFS_VALIDAS = {
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
    'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'
}
ufs_presentes = set(df['uf'].dropna().unique())
ufs_invalidas = ufs_presentes - UFS_VALIDAS
print(f"Únicas (excluindo nulos): {sorted(ufs_presentes)}")
print(f"Fora do domínio: {ufs_invalidas if ufs_invalidas else 'nenhuma'}")

# --- 6. Estatísticas: obter range e distribuição das colunas numéricas como base para o diagnóstico de outliers ---
print("\n=== Resumo estatístico ===")
display(df[['qtd', 'valor_unit', 'desconto_%', 'avaliacao']].describe())

# --- 7a. Validação de domínio: checar violações de regras de negócio com limites fixos conhecidos ---
# Método: limiar fixo. Aplicável a colunas com domínio bem definido (qtd, desconto_%, avaliacao).
# Não detecta outliers estatísticos em valor_unit, que não tem limite superior de negócio definido.
print("\n=== Validação de domínio (regras de negócio) ===")
print(f"qtd <= 0: {(df['qtd'] <= 0).sum()}")
print(f"qtd não-inteira: {((df['qtd'].dropna() % 1) != 0).sum()}")
print(f"valor_unit <= 0: {(df['valor_unit'] <= 0).sum()}")
print(f"desconto_% < 0: {(df['desconto_%'] < 0).sum()}")
print(f"desconto_% > 100: {(df['desconto_%'] > 100).sum()}")
print(f"`avaliacao` fora de [1,5]: {((df['avaliacao'] < 1) | (df['avaliacao'] > 5)).sum()}")

# --- 7b. Outliers estatísticos em valor_unit: IQR, robusto a distribuições assimétricas ---
q1_valor = df['valor_unit'].quantile(0.25)
q3_valor = df['valor_unit'].quantile(0.75)
iqr_valor = q3_valor - q1_valor
limite_inferior_valor = q1_valor - 1.5 * iqr_valor  # IQR preferível ao z-score: distribuição assimétrica (mediana R$259 << média R$484) viola normalidade
limite_superior_valor = q3_valor + 1.5 * iqr_valor  # dois limites: inferior (Q1 - 1,5×IQR) e superior (Q3 + 1,5×IQR)

registros_acima_limite = df[df['valor_unit'] > limite_superior_valor]

print(f"\n=== Outliers estatísticos: valor_unit (método IQR) ===")
print(f"Q1: R$ {q1_valor:.2f}  |  Q3: R$ {q3_valor:.2f}  |  IQR: R$ {iqr_valor:.2f}")
print(f"Limite inferior (Q1 - 1,5 × IQR): R$ {limite_inferior_valor:.2f}")
print(f"Limite superior (Q3 + 1,5 × IQR): R$ {limite_superior_valor:.2f}")
print(f"Registros abaixo do limite inferior: {(df['valor_unit'] < limite_inferior_valor).sum()}")
print(f"  Limite inferior é negativo (R$ {limite_inferior_valor:.2f}); nenhum preço pode assumir valor abaixo de zero.")
print(f"Registros acima do limite superior: {len(registros_acima_limite)}")
if not registros_acima_limite.empty:
    print(f"Produtos envolvidos (acima do limite):")
    display(
        registros_acima_limite
        .groupby(['produto', 'categoria'])['valor_unit']
        .agg(pedidos='count', preco_unitario='first')
        .sort_values('preco_unitario', ascending=False)
        .reset_index()
    )

# --- 7c. Consistência de preço por produto: detectar dispersão de valor_unit dentro do mesmo produto ---
# Método: nunique por produto. Produto com >1 valor_unit distinto indica erro de cadastro/digitação
# (padrão típico: vírgula decimal deslocada, valor 10x). Complementa o IQR, que só pega outlier global.
print("\n=== Dispersão de valor_unit por produto ===")
dispersao_valor_unit = (
    df.dropna(subset=['produto', 'valor_unit'])
      .groupby('produto')['valor_unit']
      .agg(precos_distintos='nunique', valor_min='min', valor_max='max', registros='count')
)
produtos_divergentes = dispersao_valor_unit[dispersao_valor_unit['precos_distintos'] > 1].copy()
print(f"Produtos com mais de um valor_unit: {len(produtos_divergentes)}")
if not produtos_divergentes.empty:
    produtos_divergentes['razao_max_min'] = (produtos_divergentes['valor_max'] / produtos_divergentes['valor_min']).round(2)
    display(produtos_divergentes.sort_values('razao_max_min', ascending=False))

# --- 8. Cruzamento avaliacao x status: verificar se apenas pedidos entregues possuem avaliação preenchida ---
print("\n=== Avaliação por status ===")
avaliacao_por_status = df.groupby('status', observed=True).agg(
    total_pedidos=('pedido_id', 'count'),
    com_avaliacao=('avaliacao', 'count')
)
avaliacao_por_status['pct_avaliados'] = (
    avaliacao_por_status['com_avaliacao'] / avaliacao_por_status['total_pedidos'] * 100
).round(1)
display(avaliacao_por_status)

# --- 9. Datas: detectar formatos inconsistentes que o pandas converterá em NaT e excluirá da análise temporal ---
print("\n=== data_pedido ===")
df['data_pedido_parsed'] = pd.to_datetime(df['data_pedido'], errors='coerce')
datas_invalidas = df['data_pedido_parsed'].isna().sum()

print(f"  Válidas: {len(df) - datas_invalidas}  |  NaT: {datas_invalidas}  |  Range: {df['data_pedido_parsed'].min().date()} a {df['data_pedido_parsed'].max().date()}")
print(f"\nRegistros com formato inválido (dd/mm/yyyy) — {datas_invalidas} no total:")
display(
    df.loc[df['data_pedido_parsed'].isna(), ['pedido_id', 'data_pedido']]
    .sort_values('pedido_id')
    .reset_index(drop=True)
)

df.drop(columns=['data_pedido_parsed'], inplace=True)

=== Shape ===
(1205, 11)

=== Tipos ===
pedido_id        int64
data_pedido        str
cliente_id         str
uf                 str
produto            str
categoria          str
qtd            float64
valor_unit     float64
desconto_%     float64
status             str
avaliacao      float64
dtype: object

=== Amostra ===


,pedido_id,data_pedido,cliente_id,uf,produto,categoria,qtd,valor_unit,desconto_%,status,avaliacao
0,1102,2024-03-02,C191,MG,HD Externo 2TB,Armazenamento,2.0,349.0,0.0,entregue,4.0
1,1947,2024-04-14,C036,PA,Toner HP,Impressoras,1.0,129.0,0.0,entregue,5.0
2,1307,2024-12-07,C031,MG,Mouse Gamer,Periféricos,1.0,89.0,10.0,em_transito,NaN
3,1110,2024-08-07,C079,MG,Headset 7.1,Periféricos,1.0,349.0,20.0,entregue,5.0
4,2062,2024-02-15,C119,SP,Suporte Notebook,Acessórios,2.0,89.9,0.0,entregue,4.0



=== Nulos absolutos e porcentagem ===


,nulos,percentual_%
cliente_id,25,2.07
uf,15,1.24
qtd,12,1.00
desconto_%,30,2.49
avaliacao,241,20.00



=== Missing disfarçado ===
  cliente_id: 25
  uf: 15

=== Duplicatas em pedido_id ===
Linhas duplicadas: 10  |  pedido_ids afetados: 5


...

,precos_distintos,valor_min,valor_max,registros,razao_max_min
produto,,,,,
HD Externo 2TB,2,349.0,3490.0,59,10.0
Headset 7.1,2,349.0,3490.0,68,10.0
"Monitor 24""",2,899.0,8990.0,73,10.0
Mousepad XL,2,59.9,599.0,60,10.0
SSD 500GB,2,189.0,1890.0,52,10.0
Suporte Notebook,2,89.9,899.0,63,10.0



=== Avaliação por status ===


,total_pedidos,com_avaliacao,pct_avaliados
status,,,
Devolvido,2,2,100.0
Entregue,8,8,100.0
cancelado,138,0,0.0
devolvido,82,82,100.0
em_transito,103,0,0.0
entregue,872,872,100.0



=== data_pedido ===
  Válidas: 1185  |  NaT: 20  |  Range: 2024-01-01 a 2024-12-31

Registros com formato inválido (dd/mm/yyyy) — 20 no total:


,pedido_id,data_pedido
0,1022,06/06/2024
1,1054,08/12/2024
2,1107,01/09/2024
3,1115,31/01/2024
4,1197,24/02/2024
5,1208,29/03/2024
6,1209,09/07/2024
7,1478,29/01/2024
8,1604,20/09/2024
9,1696,18/11/2024


### Achados - Q1

#### Achado principal

O dataset tem `1.205` linhas e `11` colunas, e utilizável, mas contém problemas de qualidade que distorcem análises de receita, volume, satisfação e cobertura geográfica.

---

#### Conciliação `[CODE]` vs `[MD analise]`

| Afirmação no MD | Evidência no output do `[CODE]` | Status |
|---|---|---|
| `avaliacao` com `241` ausências (`20,00%`) | tabela de nulos (`241`, `20.00`) | explícito |
| Ausências de `avaliacao` em `cancelado` (`138`) e `em_transito` (`103`) | distribuição de `status` + `Avaliação por status` | explícito |
| Média `3,92/5`, desvio `1,16`, base `964` | `describe`: `3.920124`, `1.155981`, `964` | derivado direto |
| `6` produtos com razão `10x` em `valor_unit` | bloco 7c: `Produtos com mais de um valor_unit: 6` + `razao_max_min = 10.0` | explícito |
| "`3` registros adicionais abaixo do limite estatístico" | não há contagem explícita no output para esses casos | não visível |
| "Total `8` registros com `valor_unit` anômalo" | não há total explícito no output consolidando todos os casos | não visível |

**O que é relevante mencionar para o contexto de negócio**

- Fragmentação de `status`: `Entregue` (`8`) e `Devolvido` (`2`) coexistem com versões em lowercase, isso altera KPIs por agrupamento textual.
- Outlier estatístico isolado não basta: há `118` registros acima do limite IQR, mas `113` pertencem a produtos de ticket alto (Monitor 32" 4K: `57`, Monitor 27" 144Hz: `56`), sobrando `5` suspeitos acima do limite para correção de cadastro.
- Datas inválidas (`20`) viram `NaT`, isso remove linhas de análises temporais se não houver tratamento prévio.

---

#### Evidências-chave

**Campos em branco**

| Coluna | Ausências | % | Impacto downstream |
|---|---|---|---|
| `avaliacao` | `241` | `20,00%` | Ausências concentradas em `cancelado` (`138`) e `em_transito` (`103`), análises de satisfação exigem filtro por `status` antes de calcular médias |
| `desconto_%` | `30` | `2,49%` | Pedidos sem desconto cadastrado ficam fora de qualquer cálculo de margem líquida |
| `cliente_id` | `25` | `2,07%` | Pedidos anônimos não entram em segmentação nem análise comportamental por cliente |
| `uf` | `15` | `1,24%` | Pedidos sem UF ficam fora de relatórios regionais |
| `qtd` | `12` | `1,00%` | Linhas sem quantidade não contribuem para volume nem receita |

Strings vazias confirmaram as mesmas `25` e `15` ausências, sem ocultas adicionais. `desconto_%` não apresentou violações de domínio e o máximo observado foi `20%`. Os `964` pedidos com avaliação têm média `3,92/5` e desvio padrão `1,16`.

**Pedidos duplicados**

`10` linhas correspondem a `5` pedidos distintos, contagens e somatórios de receita ficam inflados. Dois casos críticos: pedido `1113` com `cliente_id` divergente entre cópias e pedido `1885` com duplicata junto de inconsistência de categoria (`"Monitores"` vs `"monitores"`).

**Grafias inconsistentes**

`status` tem `10` registros com inicial maiúscula (`Entregue`: `8`, `Devolvido`: `2`), isso fragmenta agrupamentos. `categoria` também tem variantes em minúsculas e duplica grupos sem erro de execução.

**Quantidades inválidas e datas mal formadas**

Há `5` registros com `qtd <= 0` (mínimo `-1`). Em datas, `20` registros estão em `dd/mm/yyyy` e viram `NaT` no parse padrão. Faixa válida observada: `2024-01-01` a `2024-12-31`.

**Erros de preço unitário por produto**

No diagnóstico IQR, o limite superior de `valor_unit` é `R$ 1.304,00` e há `118` registros acima desse limite. Desse total, `113` são de produtos de ticket alto (Monitor 32" 4K `57` + Monitor 27" 144Hz `56`) e `5` ficam concentrados em padrões típicos de cadastro `10x` acima do limite (Monitor 24", HD Externo 2TB, Headset 7.1, SSD 500GB).

No diagnóstico intra-produto (bloco 7c), há `6` produtos com dois `valor_unit` e razão `10x`: Monitor 24", HD Externo 2TB, Headset 7.1, SSD 500GB, Mousepad XL e Suporte Notebook. Para os dois últimos, o output do bloco 7c confirma a divergência `10x`, mas não expõe contagem explícita de linhas infladas no mesmo quadro.

---

#### O que isso significa para o negócio

1. Receita pode ser superestimada, porque `5` pedidos distintos estão duplicados no nível de linha.
2. Relatórios regionais perdem cobertura, porque `15` pedidos sem `uf` não entram em análises por estado.
3. Indicadores de satisfação ficam enviesados se não houver filtro por `status`, pois há `241` ausências concentradas em pedidos não entregues.
4. Agrupamentos por texto podem quebrar silenciosamente por variação de capitalização em `status` e `categoria`.

---

#### Ressalvas

- O output confirma `13` UFs presentes na base e nenhuma UF fora do domínio esperado.
- O diagnóstico descreve o dataset como exportado, não permite inferir com certeza a origem operacional de cada problema.

---

#### Próximo passo, Q2

Os achados seguem catalogados em `memory-bank/data-issues.md` (DI-001 a DI-012). Ordem de tratamento recomendada:

1. **DI-002**: remover duplicatas de `pedido_id`, para `1113`, preservar a cópia com `cliente_id` preenchido.
2. **DI-012**: corrigir inconsistências de `valor_unit` com padrão `10x`, priorizando casos explicitamente suspeitos no IQR.
3. **DI-003**: excluir ou corrigir `qtd <= 0`.
4. **DI-008 / DI-009**: normalizar capitalização de `status` e `categoria`.
5. **DI-010**: reprocessar `data_pedido` suportando ambos os formatos.
6. **DI-001 / DI-006 / DI-007**: definir estratégia para campos com ausência (exclusão, marcação ou imputação).

# Bloco 3: Q2 - Tratamento e Justificativa

Q2 trata os `12` problemas catalogados como `DI-001` a `DI-012` a partir do dataset bruto `data/raw/ecommerce_vendas.csv` (imutável). `DI` significa `Data Issue`, ou problema de qualidade de dados. Por exemplo, `DI-002` é o segundo problema catalogado: duplicatas de `pedido_id`. O dataset tratado é salvo em `data/interim/ecommerce_tratado.csv`.

**Ordem de tratamento**, da prioridade alta para a baixa:

1. **DI-002** (H): duplicatas de `pedido_id`, remover repetidos preservando a cópia com `cliente_id` preenchido.
2. **DI-012** (H): `valor_unit` com padrão `10x`, corrigir dividindo por `10` para `6` produtos identificados.
3. **DI-003** (H): `qtd <= 0`, remover `5` registros inválidos.
4. **DI-008** (M): `status` com inicial maiúscula, padronizar para minúsculas (`str.lower()`).
5. **DI-009** (M): `categoria` com variantes em minúsculas, padronizar para title case (`str.title()`).
6. **DI-010** (M): `data_pedido` em `dd/mm/yyyy`, reprocessar aceitando os dois formatos.
7. **DI-007** (M): `qtd` nulo, remover `12` linhas e converter `qtd` para `Int64` (**DI-011**).
8. **DI-001** (H): `cliente_id` nulo, manter com a flag `cliente_anonimo`.
9. **DI-004** (M): `desconto_%` nulo, preencher com `0` quando não há desconto registrado.
10. **DI-005** (M): `avaliacao` nula, manter como `NaN` porque isso é esperado em `cancelado` e `em_transito`.
11. **DI-006** (M): `uf` nula, manter como `NaN` porque não há dado para preencher.
12. Renomear `desconto_%` para `desconto_pct` e criar a coluna `receita`.

In [3]:
# ============================================================
# Q2 - Tratamento de qualidade
# ============================================================
print(f"Shape inicial: {df.shape}")

# --- DI-002: Remover duplicatas de pedido_id ---
# Ordenar para que cliente_id preenchido venha antes de NaN (pedido 1113)
linhas_antes_dedup = len(df)
df = (df
      .sort_values(['pedido_id', 'cliente_id'], na_position='last')
      .drop_duplicates(subset='pedido_id', keep='first')
      .reset_index(drop=True))
print(f"\nDI-002 | Duplicatas removidas: {linhas_antes_dedup - len(df)} | Shape: {df.shape}")
print(f"  Pedido 1113 cliente_id após dedup: {df.loc[df['pedido_id'] == 1113, 'cliente_id'].values}")

# --- DI-012: Corrigir valor_unit com padrão 10x ---
# Para cada um dos 6 produtos, o menor valor_unit é o correto; os inflados têm razão exata 10x.
produtos_10x = ['HD Externo 2TB', 'Headset 7.1', 'Monitor 24"', 'Mousepad XL', 'SSD 500GB', 'Suporte Notebook']
precos_corrigidos = 0
print(f"\nDI-012 | valor_unit range antes: R${df['valor_unit'].min():.2f} a R${df['valor_unit'].max():.2f}")
print(f"DI-012 | Correção de valor_unit 10x:")
for produto in produtos_10x:
    preco_min = df.loc[df['produto'] == produto, 'valor_unit'].min()
    preco_inflado = round(preco_min * 10, 2)
    pedidos_preco_inflado = (df['produto'] == produto) & (df['valor_unit'].round(2) == preco_inflado)
    contagem_inflados = pedidos_preco_inflado.sum()  # contagem local por produto, acumulada em precos_corrigidos
    df.loc[pedidos_preco_inflado, 'valor_unit'] = preco_min
    precos_corrigidos += contagem_inflados
    print(f"  {produto}: {contagem_inflados} registro(s) de R${preco_inflado:.2f} → R${preco_min:.2f}")
print(f"DI-012 | Total corrigidos: {precos_corrigidos}")
print(f"  valor_unit range após correção: R${df['valor_unit'].min():.2f} a R${df['valor_unit'].max():.2f}")

# --- DI-003: Remover registros com qtd <= 0 ---
linhas_antes_qtd_invalida = len(df)
pedidos_qtd_invalida = df['qtd'] <= 0
print(f"\nDI-003 | Registros com qtd <= 0: {pedidos_qtd_invalida.sum()}")
df = df[~pedidos_qtd_invalida].reset_index(drop=True)
print(f"DI-003 | Removidos: {linhas_antes_qtd_invalida - len(df)} | Shape: {df.shape}")

# --- DI-008: Normalizar status para minúsculas ---
registros_status_maiusculo = (df['status'] != df['status'].str.lower()).sum()
df['status'] = df['status'].str.lower()
print(f"\nDI-008 | Registros normalizados em status: {registros_status_maiusculo}")
display(df['status'].value_counts())

# --- DI-009: Normalizar categoria para title case ---
registros_categoria_minuscula = (df['categoria'] != df['categoria'].str.title()).sum()
df['categoria'] = df['categoria'].str.title()
print(f"\nDI-009 | Registros normalizados em categoria: {registros_categoria_minuscula}")
display(df['categoria'].value_counts())

# --- DI-010: Reprocessar data_pedido suportando yyyy-mm-dd e dd/mm/yyyy ---
# Regex converte dd/mm/yyyy → yyyy-mm-dd antes do parse único
registros_formato_dmy = df['data_pedido'].str.match(r'^\d{2}/\d{2}/\d{4}$').sum()
print(f"\nDI-010 | Registros em dd/mm/yyyy (antes): {registros_formato_dmy}")
df['data_pedido'] = pd.to_datetime(
    df['data_pedido'].str.replace(
        r'^(\d{2})/(\d{2})/(\d{4})$', r'\3-\2-\1', regex=True
    ),
    format='%Y-%m-%d'
)
datas_invalidas_restantes = df['data_pedido'].isna().sum()
print(f"DI-010 | NaT após reprocessamento: {datas_invalidas_restantes}")
print(f"  Range: {df['data_pedido'].min().date()} a {df['data_pedido'].max().date()}")

# --- DI-007 + DI-011: Remover qtd nulos e converter para Int64 ---
linhas_antes_qtd_nulo = len(df)
df = df.dropna(subset=['qtd']).reset_index(drop=True)
df['qtd'] = df['qtd'].astype('Int64')
print(f"\nDI-007 | qtd nulos removidos: {linhas_antes_qtd_nulo - len(df)} | Shape: {df.shape}")
print(f"DI-011 | qtd dtype: {df['qtd'].dtype}")

# --- DI-001: Manter cliente_id nulos com flag ---
# 25 em Q1 → 24 após dedup/remoções; 1 pedido anônimo descartado junto com linhas inválidas
df['cliente_anonimo'] = df['cliente_id'].isna()
print(f"\nDI-001 | Pedidos anônimos (cliente_id nulo): {df['cliente_anonimo'].sum()} — mantidos com flag cliente_anonimo")

# --- DI-004: Imputar desconto_% nulos com 0 ---
# 30 nulos em Q1 → 2 removidos com linhas inválidas (DI-002/DI-003/DI-007) → 28 imputados aqui
desconto_nulos = df['desconto_%'].isna().sum()
df['desconto_%'] = df['desconto_%'].fillna(0.0)
print(f"\nDI-004 | desconto_% imputados com 0: {desconto_nulos}")

# --- DI-005 / DI-006: Manter ausências estruturais ---
print(f"\nDI-005 | avaliacao nulos mantidos: {df['avaliacao'].isna().sum()} (estrutural: cancelado/em_transito)")
print(f"DI-006 | uf nulos mantidos: {df['uf'].isna().sum()} (sem dado para imputar — excluídos de análises geográficas)")

# --- Renomear desconto_% → desconto_pct (nome canônico downstream) ---
df = df.rename(columns={'desconto_%': 'desconto_pct'})

# --- Derivar coluna receita ---
# qtd é Int64 (nullable); multiplicação produz Float64 — verificar NA antes de usar range
df['receita'] = (df['qtd'] * df['valor_unit'] * (1 - df['desconto_pct'] / 100)).round(2)
receita_nulos = df['receita'].isna().sum()
print(f"\nReceita derivada — range: R${df['receita'].min():.2f} a R${df['receita'].max():.2f} | NA: {receita_nulos}")

# --- Resumo final ---
print(f"\n=== Shape final: {df.shape} ===")
print(f"Diff: 1205×11 → {df.shape[0]}×{df.shape[1]}")
print("\n=== Tipos finais ===")
print(df.dtypes)
print("\n=== Nulos restantes ===")
nulos_restantes = df.isnull().sum()
display(pd.DataFrame({'nulos': nulos_restantes[nulos_restantes > 0]}))
print(f"  desconto_pct NA: {df['desconto_pct'].isna().sum()} | receita NA: {df['receita'].isna().sum()}")

# --- Salvar dataset tratado ---
INTERIM_PATH = INTERIM_DIR / 'ecommerce_tratado.csv'
df.to_csv(INTERIM_PATH, index=False, encoding='utf-8')
print(f"\nDataset tratado salvo em: {INTERIM_PATH}")

Shape inicial: (1205, 11)

DI-002 | Duplicatas removidas: 5 | Shape: (1200, 11)
  Pedido 1113 cliente_id após dedup: <StringArray>
['C019']
Length: 1, dtype: str

DI-012 | valor_unit range antes: R$39.90 a R$8990.00
DI-012 | Correção de valor_unit 10x:
  HD Externo 2TB: 1 registro(s) de R$3490.00 → R$349.00
  Headset 7.1: 2 registro(s) de R$3490.00 → R$349.00
  Monitor 24": 1 registro(s) de R$8990.00 → R$899.00
  Mousepad XL: 1 registro(s) de R$599.00 → R$59.90
  SSD 500GB: 1 registro(s) de R$1890.00 → R$189.00
  Suporte Notebook: 2 registro(s) de R$899.00 → R$89.90
DI-012 | Total corrigidos: 8
  valor_unit range após correção: R$39.90 a R$2499.00

DI-003 | Registros com qtd <= 0: 5
DI-003 | Removidos: 5 | Shape: (1195, 11)

DI-008 | Registros normalizados em status: 10


status
entregue       873
cancelado      137
em_transito    101
devolvido       84
Name: count, dtype: int64


DI-009 | Registros normalizados em categoria: 14


categoria
Periféricos      270
Acessórios       231
Armazenamento    219
Monitores        183
Impressoras      171
Câmeras          121
Name: count, dtype: int64


DI-010 | Registros em dd/mm/yyyy (antes): 20
DI-010 | NaT após reprocessamento: 0
  Range: 2024-01-01 a 2024-12-31

DI-007 | qtd nulos removidos: 12 | Shape: (1183, 11)
DI-011 | qtd dtype: Int64

DI-001 | Pedidos anônimos (cliente_id nulo): 24 — mantidos com flag cliente_anonimo

DI-004 | desconto_% imputados com 0: 28

DI-005 | avaliacao nulos mantidos: 236 (estrutural: cancelado/em_transito)
DI-006 | uf nulos mantidos: 15 (sem dado para imputar — excluídos de análises geográficas)

Receita derivada — range: R$31.92 a R$11870.25 | NA: 0

=== Shape final: (1183, 13) ===
Diff: 1205×11 → 1183×13

=== Tipos finais ===
pedido_id                   int64
data_pedido        datetime64[us]
cliente_id                    str
uf                            str
produto                       str
categoria                     str
qtd                         Int64
valor_unit                float64
desconto_pct              float64
status                        str
avaliacao                 float64
cl

,nulos
cliente_id,24
uf,15
avaliacao,236


  desconto_pct NA: 0 | receita NA: 0

Dataset tratado salvo em: ..\data\interim\ecommerce_tratado.csv


## Resultado do tratamento

O dataset passou de `1.205 × 11` para `1.183 × 13`. Foram removidas `22` linhas, todas sem possibilidade de recuperação confiável, e adicionadas `2` colunas derivadas: `cliente_anonimo` e `receita`. O campo `desconto_%` foi renomeado para `desconto_pct`, que será o nome padrão nas próximas análises.

**Remoções por etapa:**

| Etapa | Linhas removidas | Shape após |
|---|---|---|
| DI-002: remoção de duplicatas de `pedido_id` | `5` | `1.200 × 11` |
| DI-003: `qtd <= 0` | `5` | `1.195 × 11` |
| DI-007: `qtd` nulo | `12` | `1.183 × 11` |

---

## Decisões aplicadas e impacto no negócio

### DI-002: Duplicatas de `pedido_id` (`5` linhas removidas)

Cinco `pedido_id` apareciam repetidos, o que aumentaria artificialmente a contagem de pedidos e o faturamento. No pedido `1113`, uma cópia tinha `cliente_id = C019` e a outra estava sem `cliente_id`. A regra usada manteve a cópia com cliente identificado e descartou a repetida. Os outros quatro pedidos repetidos tinham cópias iguais.

### DI-012: `valor_unit` com padrão `10×` (`8` registros corrigidos, `6` produtos)

Em `6` produtos, o maior preço era exatamente `10` vezes o menor (`max/min = 10,0`). Isso indica erro de cadastro, provavelmente vírgula decimal no lugar errado, e não uma mudança normal de preço. Por isso, os `8` registros com preço inflado foram corrigidos dividindo `valor_unit` por `10`. Cada linha afetada fazia a receita parecer até `9×` maior que o valor real. O intervalo de `valor_unit` saiu de `R$39,90` a `R$8.990,00` e ficou em `R$39,90` a `R$2.499,00`.

Registros corrigidos por produto:

| Produto | Registros | Preço inflado | Preço correto |
|---|---|---|---|
| HD Externo 2TB | `1` | `R$3.490,00` | `R$349,00` |
| Headset 7.1 | `2` | `R$3.490,00` | `R$349,00` |
| Monitor 24" | `1` | `R$8.990,00` | `R$899,00` |
| Mousepad XL | `1` | `R$599,00` | `R$59,90` |
| SSD 500GB | `1` | `R$1.890,00` | `R$189,00` |
| Suporte Notebook | `2` | `R$899,00` | `R$89,90` |

A causa dos erros não foi identificada nesta análise. O padrão foi corrigido, mas a origem operacional continua sendo um ponto que os dados não explicam.

### DI-003: `qtd <= 0` (`5` removidos)

Quantidade negativa ou zero não representa uma venda real. Como não há base confiável para preencher outro valor, as `5` linhas foram removidas. Isso evita que o volume de unidades vendidas seja calculado com transações inválidas.

### DI-008: `status` com inicial maiúscula (`10` normalizados)

`Entregue` (`8`) e `Devolvido` (`2`) foram padronizados para minúsculas. Sem essa correção, filtros e agrupamentos por `status` separariam valores que significam a mesma coisa. A lista final de valores ficou: `entregue` (`873`), `cancelado` (`137`), `em_transito` (`101`) e `devolvido` (`84`).

### DI-009: `categoria` com variantes em minúsculas (`14` normalizados)

Quatorze registros com categoria em minúsculas foram padronizados para title case, ou seja, primeira letra maiúscula em cada palavra. O total ficou em `14`, e não `15` como visto em Q1, porque um registro em minúsculas foi removido junto com uma duplicata de `pedido_id` em DI-002. A lista final de categorias ficou: `Periféricos` (`270`), `Acessórios` (`231`), `Armazenamento` (`219`), `Monitores` (`183`), `Impressoras` (`171`) e `Câmeras` (`121`).

### DI-010: `data_pedido` em `dd/mm/yyyy` (`20` corrigidos, `0` NaT restantes)

Os `20` registros em `dd/mm/yyyy` foram convertidos antes da leitura das datas. O resultado foi `0` datas inválidas e cobertura completa de `2024-01-01` a `2024-12-31`, sem precisar excluir pedidos por problema de formato.

### DI-007 + DI-011: `qtd` nulo e tipo (`12` removidos, dtype `Int64`)

Doze linhas sem quantidade foram removidas, porque a receita não pode ser calculada sem `qtd`. Preencher esse campo por estimativa seria uma escolha sem evidência. Depois disso, `qtd` foi convertido de `float64` para `Int64`, um tipo inteiro, para evitar casas decimais falsas em quantidades.

### DI-001: `cliente_id` nulo (`24` mantidos com flag `cliente_anonimo`)

Os `24` pedidos sem `cliente_id` representam `2,0%` da base após remover duplicatas e têm receita real. Removê-los reduziria volume e faturamento sem necessidade. A coluna `cliente_anonimo` marca esses casos, então análises de segmentação e comportamento por cliente devem filtrar `cliente_anonimo == False` antes de calcular métricas por cliente.

### DI-004: `desconto_pct` nulo (`28` preenchidos com `0`)

Os `28` registros sem desconto foram tratados como pedidos sem desconto aplicado. Essa é uma escolha conservadora, porque não aumenta artificialmente o desconto. O impacto real desse preenchimento nos `28` registros não foi verificado e segue como ponto que os dados não explicam.

### DI-005: `avaliacao` nulo (`236` mantidos como `NaN`)

Os `236` nulos são esperados pelo desenho do dado: apenas pedidos `entregue` ou `devolvido` têm avaliação. Preencher satisfação para pedidos `cancelado` ou `em_transito` criaria uma nota falsa. Análises de satisfação devem usar somente registros com `avaliacao` preenchida.

### DI-006: `uf` nulo (`15` mantidos como `NaN`)

Sem dado externo para descobrir a UF, criar um estado fictício distorceria análises regionais. Os `15` pedidos ficam fora de relatórios geográficos, e a cobertura declarada é `98,7%` da base.

---

## Derivações

A coluna `receita` foi criada como métrica calculada, não como campo original do dataset. Ela usa `qtd`, `valor_unit` e `desconto_pct` já tratados para padronizar a receita líquida usada nas próximas análises: `receita = qtd × valor_unit × (1 - desconto_pct / 100)`. O motivo prático é que várias análises posteriores dependem de receita líquida, especialmente nas Questões 3 e 5. Por exemplo, a Questão 3 pede os produtos com maior receita líquida após desconto. Se essa coluna não fosse criada no dataset tratado, cada análise teria que recalcular a mesma fórmula, aumentando o risco de inconsistência. Após o cálculo, o intervalo de `receita` ficou entre `R$31,92` e `R$11.870,25`, sem nenhum `NA` em `receita`.

---

## Status do dataset tratado: aprovado com ressalvas

O dataset `data/interim/ecommerce_tratado.csv` (`1.183 × 13`) é o insumo confiável para as próximas análises. Os `12` problemas catalogados foram corrigidos ou documentados.

**Nulos remanescentes intencionais:**

| Coluna | Nulos | % | Ressalva operacional |
|---|---|---|---|
| `avaliacao` | `236` | `19,9%` | Calcular somente sobre `entregue` e `devolvido` |
| `cliente_id` | `24` | `2,0%` | Filtrar `cliente_anonimo == False` antes de segmentação |
| `uf` | `15` | `1,3%` | Declarar cobertura de `98,7%` em análises regionais |

# Bloco 4: Q3 - SQL

## Q3 - Consultas SQL

Quatro consultas respondem perguntas recorrentes do comercial sobre o dataset tratado em Q2. Padrão do bloco:

- DuckDB in-memory via `duckdb.connect()`, conexão única reaproveitada por todas as subquestões.
- View `vendas` registrada a partir do `df` tratado em memória (equivalente ao artefato persistente `data/interim/ecommerce_tratado.csv`), sem reler o CSV.
- Colunas canônicas: `desconto_pct` (renomeada em Q2 a partir de `desconto_%`) e `receita` (derivada em Q2 como `qtd * valor_unit * (1 - desconto_pct / 100)`). Agregações de receita usam `SUM(receita)` para manter Q2 como única fonte da fórmula.
- Cada query vive em `sql/q3_*.sql` e é carregada no notebook via `con.sql(open(...).read()).df()`, garantindo que o arquivo `.sql` seja reproduzível isoladamente.

### Q3.a Top 5 produtos por receita líquida nos últimos 90 dias

**Pergunta:** quais são os `5` produtos com maior receita líquida, considerando apenas pedidos concluídos na janela final do dataset?

**Métrica:** receita líquida agregada por produto via `SUM(receita)`.

**Filtros:**
- `status NOT IN ('cancelado', 'devolvido')`: pedidos cancelados e devolvidos não representam receita realizada.
- `data_pedido >= MAX(data_pedido) - INTERVAL 90 DAY`: janela ancorada na data mais recente do dataset, não em `CURRENT_DATE`. Como o range vai até `2024-12-31` (confirmado em Q2), a janela efetiva é o quarto trimestre.

**Saída:** ranking com `produto`, `receita_liquida`, `unidades_vendidas` e `pedidos`, ordenado por `receita_liquida DESC`, limitado a `5` linhas.

In [4]:
# Q3 - Setup: conexao DuckDB in-memory reusada por todas as subquestoes
con = duckdb.connect()
con.register("vendas", df)

# Q3.a - Top 5 produtos por receita liquida nos ultimos 90 dias
sql_q3a = (SQL_DIR / "q3_a_top5_receita.sql").read_text(encoding="utf-8")
top5_receita_90d = con.sql(sql_q3a).df()

data_max = df["data_pedido"].max().date()
print(f"Ancora temporal: MAX(data_pedido) = {data_max} | Janela: ultimos 90 dias")
display(top5_receita_90d)

Ancora temporal: MAX(data_pedido) = 2024-12-31 | Janela: ultimos 90 dias


,produto,receita_liquida,unidades_vendidas,pedidos
0,"Monitor 32"" 4K",55977.60,24,11
1,Impressora Laser,28471.50,31,18
2,"Monitor 24""",25981.10,31,19
3,"Monitor 27"" 144Hz",25108.25,18,13
4,Impressora Jato,11860.20,21,12


### Achados - Q3.a

**Âncora temporal:** `MAX(data_pedido) = 2024-12-31`. A janela de `90` dias cobre `2024-10-02` a `2024-12-31`, correspondendo aproximadamente ao Q4 do dataset.

**Top 5 produtos por receita líquida:**

| Posição | Produto | Receita líquida | Unidades | Pedidos |
|---|---|---|---|---|
| `1` | Monitor 32" 4K | `R$ 55.977,60` | `24` | `11` |
| `2` | Impressora Laser | `R$ 28.471,50` | `31` | `18` |
| `3` | Monitor 24" | `R$ 25.981,10` | `31` | `19` |
| `4` | Monitor 27" 144Hz | `R$ 25.108,25` | `18` | `13` |
| `5` | Impressora Jato | `R$ 11.860,20` | `21` | `12` |

---

#### Padrão dominante, monitores e impressoras

Três monitores e duas impressoras formam o top 5. Nenhum produto de `Periféricos`, `Acessórios`, `Armazenamento` ou `Câmeras` aparece no ranking. A leitura direta é que a receita líquida no Q4 é puxada por ticket médio alto, não por volume de pedidos.

#### Concentração forte no líder

`Monitor 32" 4K` sozinho gerou `R$ 55.977,60`, quase duas vezes o segundo colocado (`R$ 28.471,50`). O ticket por pedido, ou seja, receita líquida dividida pelo número de pedidos, é o maior do ranking: `55.977,60 / 11 ≈ R$ 5.088,87`. O produto combina preço unitário alto e compra com quantidade, com média de `24 / 11 ≈ 2,2` unidades por pedido, a única do ranking acima de `2` unidades médias.

#### Volume semelhante, receita diferente

`Impressora Laser` e `Monitor 24"` vendem praticamente o mesmo volume de unidades (`31` cada) e têm contagens de pedidos próximas (`18` e `19`), mas a `Impressora Laser` supera em receita líquida (`R$ 28.471,50` vs. `R$ 25.981,10`). Em volumes equivalentes, o ticket unitário é o que decide a posição no ranking.

---

#### O que isso significa para o negócio

A diretoria pediu para identificar categorias de "alto desempenho" para dobrar investimento. Na janela de `90` dias, `Monitores` e `Impressoras` concentram a receita líquida, não as categorias de maior volume. Decisões de estoque, campanhas e foco comercial orientadas por receita devem começar por esses dois grupos. Em particular, `Monitor 32" 4K` é o SKU com maior peso isolado, portanto ruptura de estoque ou promoção específica desse produto tem impacto desproporcional no faturamento trimestral.

---

#### Ressalvas

- **Janela curta:** `90` dias equivalem a Q4/2024 e não representam o ano inteiro. Sazonalidade de fim de ano pode inflar monitores e impressoras em relação a trimestres anteriores, e o ranking não confirma alto desempenho fora dessa janela.
- **Amostra pequena por produto:** cada linha do top 5 tem entre `11` e `19` pedidos. Um pedido atípico grande muda a posição com facilidade, especialmente em `Monitor 32" 4K` com apenas `11` pedidos.
- **Receita realizada:** o ranking já exclui `cancelado` e `devolvido` via filtro de `status`, portanto não precisa ser ajustado por taxa de cancelamento posterior. Receita bruta (incluindo cancelados) e taxa de cancelamento por categoria ficam para Q3.b.

### Q3.b Taxa de cancelamento e devolução por categoria

**Pergunta:** em cada categoria, qual a fração de pedidos que não se converteu em receita realizada, somando `cancelado` e `devolvido`?

**Métrica:** `taxa_cancelamento_devolucao_pct = 100 × pedidos_nao_concluidos / total_pedidos`, calculada por categoria. Taxa combinada, conforme pedido do PRD, com `status IN ('cancelado', 'devolvido')` no numerador.

**Base:**
- Denominador: todos os pedidos da categoria, sem filtro de status.
- Sem janela temporal, usa o dataset tratado inteiro (`2024-01-01` a `2024-12-31`).
- Categorias já normalizadas em title case em Q2 (DI-009), sem fragmentação por capitalização.

**Saída:** `categoria`, `total_pedidos`, `nao_concluidos` e `taxa_cancelamento_devolucao_pct`, ordenada por taxa do maior para o menor.

In [5]:
# Q3.b - Taxa de cancelamento e devolucao por categoria
sql_q3b = (SQL_DIR / "q3_b_taxas_categoria.sql").read_text(encoding="utf-8")
taxas_cancelamento_por_categoria = con.sql(sql_q3b).df()

base_total = int(taxas_cancelamento_por_categoria["total_pedidos"].sum())
nao_concluidos_total = int(taxas_cancelamento_por_categoria["nao_concluidos"].sum())
taxa_global = round(100 * nao_concluidos_total / base_total, 2)
print(f"Base: {base_total} pedidos | Nao concluidos (cancelado + devolvido): {nao_concluidos_total} | Taxa global: {taxa_global}%")

# Sanity-check local: reconstruir a base apos DI-009 e antes de DI-007 para sustentar a ressalva.
q3b_base_pre_di007 = pd.read_csv(RAW_PATH)
q3b_base_pre_di007 = (
    q3b_base_pre_di007.sort_values(["pedido_id", "cliente_id"], na_position="last")
    .drop_duplicates(subset="pedido_id", keep="first")
    .reset_index(drop=True)
)
q3b_base_pre_di007 = q3b_base_pre_di007[
    q3b_base_pre_di007["qtd"].isna() | (q3b_base_pre_di007["qtd"] > 0)
].reset_index(drop=True)
q3b_base_pre_di007["categoria"] = q3b_base_pre_di007["categoria"].str.title()

totais_q2_pos_di009 = (
    q3b_base_pre_di007["categoria"].value_counts().sort_index().rename("total_q2_pos_di009")
)
totais_q3b_base_final = (
    taxas_cancelamento_por_categoria.set_index("categoria")["total_pedidos"]
    .sort_index()
    .rename("total_q3b_base_final")
)
comparativo_di007 = pd.concat([totais_q2_pos_di009, totais_q3b_base_final], axis=1)
comparativo_di007["delta_di007"] = (
    comparativo_di007["total_q2_pos_di009"] - comparativo_di007["total_q3b_base_final"]
)
comparativo_di007 = comparativo_di007.reset_index().rename(columns={"index": "categoria"})
delta_total_di007 = int(comparativo_di007["delta_di007"].sum())
print(f"Sanity-check DI-007: delta total explicado = {delta_total_di007} linhas")
display(comparativo_di007)
display(taxas_cancelamento_por_categoria)

Base: 1183 pedidos | Nao concluidos (cancelado + devolvido): 219 | Taxa global: 18.51%
Sanity-check DI-007: delta total explicado = 12 linhas


,categoria,total_q2_pos_di009,total_q3b_base_final,delta_di007
0,Acessórios,231,228,3
1,Armazenamento,219,215,4
2,Câmeras,121,120,1
3,Impressoras,171,169,2
4,Monitores,183,182,1
5,Periféricos,270,269,1


,categoria,total_pedidos,nao_concluidos,taxa_cancelamento_devolucao_pct
0,Periféricos,269,59,21.93
1,Monitores,182,38,20.88
2,Câmeras,120,21,17.50
3,Armazenamento,215,37,17.21
4,Acessórios,228,37,16.23
5,Impressoras,169,27,15.98


### Achados - Q3.b

**Taxa global:** `18,51%` (`219` pedidos não concluídos em `1.183` no total). Qualquer categoria acima desse patamar está pior que a média do negócio.

**Ranking por categoria (maior para menor):**

| Posição | Categoria | Total pedidos | Não concluídos | Taxa |
|---|---|---|---|---|
| `1` | Periféricos | `269` | `59` | `21,93%` |
| `2` | Monitores | `182` | `38` | `20,88%` |
| `3` | Câmeras | `120` | `21` | `17,50%` |
| `4` | Armazenamento | `215` | `37` | `17,21%` |
| `5` | Acessórios | `228` | `37` | `16,23%` |
| `6` | Impressoras | `169` | `27` | `15,98%` |

---

#### Duas faixas separadas por quase `3,4` pontos percentuais

Há um degrau claro entre as duas primeiras e as demais. `Periféricos` e `Monitores` estão acima de `20%` e acima da média global. `Câmeras`, `Armazenamento`, `Acessórios` e `Impressoras` ficam agrupadas entre `15,98%` e `17,50%`, todas abaixo da média. A amplitude total é de `5,95` pontos percentuais, ou seja, a diferença entre o topo (`21,93%`) e o fim do ranking (`15,98%`).

#### Cruzamento com Q3.a, receita versus falha operacional

`Monitores` e `Impressoras` apareceram no top `5` de receita líquida em Q3.a, mas se comportam de forma oposta aqui:

- `Monitores`: alta receita (três produtos no top `5`) combinada com a segunda maior taxa de cancelamento/devolução (`20,88%`). É a categoria com maior risco entre as líderes de faturamento. Cada ponto percentual de perda pesa mais, dado o ticket médio alto observado em Q3.a (`Monitor 32" 4K` com receita por pedido de `R$ 5.088,87`).
- `Impressoras`: também no top `5` de receita (posições `2` e `5`) e com a menor taxa do ranking (`15,98%`). É a categoria mais estável entre as que movimentam receita relevante.

`Periféricos`, por outro lado, não aparece no top `5` de receita em Q3.a (ticket unitário baixo), mas tem a maior taxa e o maior volume absoluto de pedidos não concluídos (`59` em `269`). É a categoria que mais consome operação sem retorno proporcional em receita realizada.

---

#### O que isso significa para o negócio

1. **`Monitores` precisa de investigação antes de qualquer plano de expansão.** A diretoria quer dobrar investimento em categorias de alto desempenho (contexto do PRD). `Monitores` lidera em receita mas perde uma em cada cinco vendas para cancelamento ou devolução. Causas possíveis a investigar: frete e avaria em itens volumosos, arrependimento em ticket alto, problemas de estoque. Dobrar investimento sem resolver esse ponto amplifica a perda.
2. **`Periféricos` exige saneamento operacional.** `59` pedidos perdidos é o maior número absoluto do ranking e representa o maior atrito em volume, mesmo com ticket médio baixo.
3. **`Impressoras` é a categoria de melhor relação receita versus atrito.** Se o objetivo for escalar com risco controlado, é candidata melhor que `Monitores` isoladamente.

---

#### Ressalvas

- **Cancelamento e devolução somados:** a taxa junta dois eventos operacionais distintos. Cancelamento geralmente acontece antes do envio (estoque, pagamento, logística), devolução acontece depois da entrega (defeito, arrependimento). A query atual não separa, então uma categoria com perfil de cancelamento pré-envio e outra com perfil de devolução pós-entrega aparecem com a mesma taxa.
- **Amostras desiguais:** `Câmeras` (`120` pedidos) e `Impressoras` (`169`) têm base menor que `Periféricos` (`269`) e `Acessórios` (`228`). Diferenças pequenas entre `Câmeras` (`17,50%`) e `Armazenamento` (`17,21%`) podem ser ruído de amostra, não padrão estrutural.
- **Sem janela temporal:** a taxa agrega o ano inteiro (`2024-01-01` a `2024-12-31`). Se houve mudança operacional durante o ano, por exemplo troca de transportadora ou fornecedor, a taxa anual mistura os períodos. Um recorte mensal ou trimestral mostraria tendência, mas fica fora do escopo de Q3.b.
- **Pequeno desvio vs. Q2:** os totais por categoria aqui estão `12` unidades abaixo dos valores reportados em Q2 (exemplo: `Periféricos` `269` vs. `270`). A diferença é consistente com as `12` linhas removidas em DI-007 (`qtd` nulo), distribuídas entre as categorias. Nenhuma implicação para a taxa, apenas sanity-check de rastreabilidade.

### Q3.c Clientes com pedidos em pelo menos 3 meses consecutivos

**Pergunta:** quais clientes identificados fizeram pedidos concluídos em `3` ou mais meses-calendário imediatamente consecutivos em algum momento de 2024?

**Critério de consecutividade:** meses consecutivos são meses-calendário com gap igual a `1`. Se um cliente compra em janeiro, fevereiro e março, qualifica; se compra em janeiro, março e abril, não qualifica (gap `2` entre janeiro e março quebra a sequência).

**Técnica (exigida pelo PRD):** `LAG` combinada com soma cumulativa:

1. Projeta cada data como `mes_num = ano × 12 + mês` (inteiro monotônico, robusto a virada de ano).
2. Deduplica o par `(cliente_id, mes_num)` para evitar contagem múltipla quando o cliente compra mais de uma vez no mesmo mês.
3. `LAG(mes_num)` fornece o mês anterior por cliente; a subtração produz o `gap_meses`.
4. Soma cumulativa de uma flag que incrementa a cada `gap ≠ 1` cria um `grupo_id` por sequência contínua.
5. `COUNT(*) >= 3` por `(cliente_id, grupo_id)` seleciona clientes com pelo menos uma sequência qualificada. `DISTINCT` impede dupla contagem se o cliente tiver várias sequências no ano.

**Filtros:**
- `status NOT IN ('cancelado', 'devolvido')`: só pedidos concluídos caracterizam recorrência real, consistente com Q3.a.
- `cliente_anonimo = FALSE`: sem `cliente_id` não há como atribuir pedidos ao mesmo titular. Q2 (DI-001) marcou `24` pedidos anônimos com essa flag.

**Saída:** `cliente_id`, `pedidos_totais` e `receita_total`, ordenada por `receita_total DESC`. Importante: `pedidos_totais` e `receita_total` cobrem **todos** os pedidos concluídos do cliente no ano, não apenas os meses da sequência qualificadora. Essa é a leitura útil para negócio, porque mede o valor total do cliente fiel, não só o trecho em que ele foi contínuo.

In [6]:
# Q3.c - Clientes com pedidos em pelo menos 3 meses consecutivos
sql_q3c = (SQL_DIR / "q3_c_clientes_recorrentes.sql").read_text(encoding="utf-8")
clientes_recorrentes = con.sql(sql_q3c).df()

total_clientes_identificados = df.loc[~df["cliente_anonimo"], "cliente_id"].nunique()
qtd_recorrentes = len(clientes_recorrentes)
pct_recorrentes = round(100 * qtd_recorrentes / total_clientes_identificados, 2) if total_clientes_identificados else 0.0
receita_top_2 = round(clientes_recorrentes.head(2)["receita_total"].sum(), 2)
receita_3_a_5 = round(clientes_recorrentes.iloc[2:5]["receita_total"].sum(), 2)
print(f"Clientes identificados (nao anonimos): {total_clientes_identificados} | Recorrentes 3+ meses: {qtd_recorrentes} ({pct_recorrentes}%)")
print(f"Receita top 2: R$ {receita_top_2:.2f} | Receita 3o ao 5o: R$ {receita_3_a_5:.2f}")
display(clientes_recorrentes)

Clientes identificados (nao anonimos): 200 | Recorrentes 3+ meses: 42 (21.0%)
Receita top 2: R$ 24049.53 | Receita 3o ao 5o: R$ 19716.89


,cliente_id,pedidos_totais,receita_total
0,C093,9,13415.25
1,C108,7,10634.28
2,C197,10,6615.16
3,C191,5,6604.40
4,C090,11,6497.33
5,C189,8,6318.60
6,C019,8,6091.96
7,C200,9,6020.48
8,C035,5,5982.60
9,C177,5,5644.10


### Achados - Q3.c

**Universo e taxa de recorrência:** `200` clientes identificados na base tratada e `42` qualificam como recorrentes com pelo menos `3` meses consecutivos de compra, ou seja, `21,0%` da base elegível.

**Top 5 recorrentes por receita total:**

| Posição | `cliente_id` | `pedidos_totais` | `receita_total` | Ticket médio/pedido |
|---|---|---|---|---|
| `1` | `C093` | `9` | `R$ 13.415,25` | `R$ 1.490,58` |
| `2` | `C108` | `7` | `R$ 10.634,28` | `R$ 1.519,18` |
| `3` | `C197` | `10` | `R$ 6.615,16` | `R$ 661,52` |
| `4` | `C191` | `5` | `R$ 6.604,40` | `R$ 1.320,88` |
| `5` | `C090` | `11` | `R$ 6.497,33` | `R$ 590,67` |

---

#### Concentração no topo

`C093` lidera com `R$ 13.415,25`, quase `2×` o segundo colocado (`C108`, `R$ 10.634,28`). Há uma inflexão clara entre os dois primeiros e o restante: do `3º` ao `5º` a receita fica entre `R$ 6.497,33` e `R$ 6.615,16`, faixa estreita. A amplitude total do ranking é larga: do topo (`C093`, `R$ 13.415,25`) ao último (`C032`, `R$ 698,25`), razão aproximada de `19,2×`. Somados, `C093` e `C108` entregam `R$ 24.049,53`, mais que as três posições seguintes juntas (`R$ 19.716,89`).

#### Volume de pedidos não explica receita sozinho

`C090` tem `11` pedidos (maior contagem do ranking) e fica em `5º` lugar com `R$ 6.497,33` e ticket médio de `R$ 590,67/pedido`. `C108`, com apenas `7` pedidos, ocupa `2º` com ticket médio de `R$ 1.519,18/pedido`, o maior do top 5. Recorrência frequente com ticket baixo rende menos receita que recorrência moderada com ticket alto.

#### Piso mínimo da qualificação

`C184` aparece com `3` pedidos concluídos no ano e `R$ 1.149,76` de receita total. Esse é o piso estrutural: `1` pedido por mês ao longo de `3` meses consecutivos já qualifica, pela regra de deduplicação cliente-mês exigida pela técnica `LAG`. A métrica responde "comprou em meses consecutivos", não "comprou com intensidade".

#### Conexão com Q3.a e Q3.b

O ticket médio dos dois primeiros (`R$ 1.490,58` e `R$ 1.519,18` por pedido) está na faixa dos produtos do top 5 de receita de Q3.a, especialmente `Impressora Laser` e `Monitor 24"`. Clientes recorrentes de alto valor provavelmente concentram compras nessas categorias. Como `Monitores` tem a `2ª` maior taxa de cancelamento e devolução (`20,88%`, Q3.b), perder um cliente recorrente de ticket alto por problema operacional em monitor custa mais do que perder um cliente esporádico de mesmo perfil de compra.

---

#### O que isso significa para o negócio

1. **Base concreta para programa de fidelidade:** `21,0%` de recorrência indica um núcleo identificável de clientes fiéis, com critério objetivo e reproduzível. Serve como ponto de partida para ativação segmentada e comunicação diferenciada.
2. **Concentração de risco no topo:** os dois maiores recorrentes somam mais que os três seguintes juntos. Perda isolada de `C093` ou `C108` tem impacto desproporcional; políticas de retenção devem ser priorizadas por valor, não por frequência de compra.
3. **Upsell vence frequência pura:** o contraste entre `C090` (alto volume, receita intermediária) e `C108` (volume médio, receita alta) mostra que estratégia de ticket (upsell, mix de categoria) pode gerar mais valor que apenas aumentar frequência.

---

#### Ressalvas

- **Janela de `1` ano:** o dataset cobre `2024-01-01` a `2024-12-31`. "Recorrente em `3+` meses consecutivos" é indicador anual, não consegue distinguir padrão estrutural de coincidência pontual sem histórico multi-ano.
- **Intensidade não considerada:** a definição usa deduplicação cliente-mês. Um cliente com `1` pedido/mês por `3` meses e outro com vários pedidos/mês aparecem ambos como recorrentes, embora tenham perfis distintos.
- **`pedidos_totais` e `receita_total` são do ano todo:** cobrem todos os pedidos concluídos do cliente qualificado em 2024, não apenas os meses da sequência qualificadora. Leitura útil para valor total do cliente fiel, mas não responde "quanto rendeu durante a sequência".
- **Filtro de status consistente com Q3.a:** `cancelado` e `devolvido` ficam fora do cálculo de recorrência, evitando que pedidos não concluídos inflacionem a consecutividade.
- **`cliente_anonimo = FALSE` exclui `24` pedidos:** conforme DI-001 em Q2, pedidos anônimos não são atribuíveis a um titular e ficam fora do universo de recorrência.

### Q3.d Ticket médio por UF com pedidos entregues

**Pergunta:** qual o ticket médio (`receita / pedidos`) por UF, considerando apenas pedidos que chegaram ao destino?

**Métrica:** `ticket_medio = SUM(receita) / COUNT(DISTINCT pedido_id)` por UF, em `R$/pedido`. Usa a coluna canônica `receita` derivada em Q2.

**Filtros:**
- `status = 'entregue'`: apenas pedidos concluídos. `cancelado`, `devolvido` e `em_transito` ficam fora, porque não caracterizam receita realizada no cliente final.
- `uf IS NOT NULL`: os `12` pedidos sem UF entre os entregues são excluídos conforme decisão de Q2 (DI-006: sem dado externo para imputar estado, criar UF fictícia distorceria análise regional). Cobertura geográfica sobre o universo entregue: `98,6%`.

**Base:** dataset tratado inteiro (`2024-01-01` a `2024-12-31`), sem janela temporal. Categorias não entram no agrupamento, ou seja, o ticket é uma média da "cesta de compra entregue" por estado, independente de mix de produto.

**Saída:** `uf`, `pedidos_entregues`, `receita_total` e `ticket_medio`, ordenada por `ticket_medio DESC`. O `receita_total` é colocado como contexto para distinguir UFs de ticket alto com baixo volume de UFs de ticket alto com volume relevante.

In [7]:
# Q3.d - Ticket medio por UF com pedidos entregues
sql_q3d = (SQL_DIR / "q3_d_ticket_uf.sql").read_text(encoding="utf-8")
ticket_medio_por_uf = con.sql(sql_q3d).df()

pedidos_entregues_total = int(df.loc[df["status"] == "entregue", "pedido_id"].nunique())
pedidos_entregues_sem_uf = int(df[(df["status"] == "entregue") & df["uf"].isna()]["pedido_id"].nunique())
pedidos_considerados = int(ticket_medio_por_uf["pedidos_entregues"].sum())
razao_topo_ultimo = round(ticket_medio_por_uf.iloc[0]["ticket_medio"] / ticket_medio_por_uf.iloc[-1]["ticket_medio"], 2)
participacao_sp_volume = round(100 * ticket_medio_por_uf.loc[ticket_medio_por_uf["uf"] == "SP", "pedidos_entregues"].iloc[0] / pedidos_considerados, 1)
ticket_sc = ticket_medio_por_uf.loc[ticket_medio_por_uf["uf"] == "SC", "ticket_medio"].iloc[0]
ticket_rs = ticket_medio_por_uf.loc[ticket_medio_por_uf["uf"] == "RS", "ticket_medio"].iloc[0]
razao_sc_rs = round(ticket_sc / ticket_rs, 2)
print(f"Pedidos entregues total: {pedidos_entregues_total} | Sem UF (excluidos): {pedidos_entregues_sem_uf} | Considerados: {pedidos_considerados}")
print(f"Razao topo/ultimo ticket: {razao_topo_ultimo}x | Participacao SP no volume: {participacao_sp_volume}% | Razao SC/RS: {razao_sc_rs}x")
display(ticket_medio_por_uf)

Pedidos entregues total: 863 | Sem UF (excluidos): 12 | Considerados: 851
Razao topo/ultimo ticket: 3.67x | Participacao SP no volume: 28.3% | Razao SC/RS: 2.12x


,uf,pedidos_entregues,receita_total,ticket_medio
0,SC,59,66947.83,1134.71
1,GO,19,19788.77,1041.51
2,PA,14,12833.99,916.71
3,RJ,127,113062.91,890.26
4,MG,97,78331.20,807.54
5,SP,241,182907.38,758.95
6,PR,54,40334.86,746.94
7,CE,26,19351.94,744.31
8,BA,45,29574.03,657.20
9,ES,27,15109.33,559.60


### Achados - Q3.d

**Base:** `863` pedidos entregues no dataset tratado, `12` sem UF excluídos (cobertura geográfica de `98,6%` do universo entregue), `851` considerados em `13` UFs.

**Ranking completo por ticket médio:**

| Posição | UF | `pedidos_entregues` | `receita_total` | `ticket_medio` |
|---|---|---|---|---|
| `1` | `SC` | `59` | `R$ 66.947,83` | `R$ 1.134,71` |
| `2` | `GO` | `19` | `R$ 19.788,77` | `R$ 1.041,51` |
| `3` | `PA` | `14` | `R$ 12.833,99` | `R$ 916,71` |
| `4` | `RJ` | `127` | `R$ 113.062,91` | `R$ 890,26` |
| `5` | `MG` | `97` | `R$ 78.331,20` | `R$ 807,54` |
| `6` | `SP` | `241` | `R$ 182.907,38` | `R$ 758,95` |
| `7` | `PR` | `54` | `R$ 40.334,86` | `R$ 746,94` |
| `8` | `CE` | `26` | `R$ 19.351,94` | `R$ 744,31` |
| `9` | `BA` | `45` | `R$ 29.574,03` | `R$ 657,20` |
| `10` | `ES` | `27` | `R$ 15.109,33` | `R$ 559,60` |
| `11` | `RS` | `73` | `R$ 39.010,04` | `R$ 534,38` |
| `12` | `PE` | `52` | `R$ 24.159,58` | `R$ 464,61` |
| `13` | `DF` | `17` | `R$ 5.259,06` | `R$ 309,36` |

---

#### Amplitude ampla e três faixas de ticket

Do topo (`SC`, `R$ 1.134,71`) ao último (`DF`, `R$ 309,36`), a razão é `3,67×`. Três faixas visíveis:

- **Premium (`> R$ 900`):** `SC`, `GO`, `PA`. Apenas `SC` tem volume relevante (`59`).
- **Intermediária (`R$ 600` a `R$ 900`):** `RJ`, `MG`, `SP`, `PR`, `CE`, `BA`. Concentra a massa comercial.
- **Baixa (`< R$ 600`):** `ES`, `RS`, `PE`, `DF`. Inclui estados de volume médio (`RS`, `PE`) e baixo (`ES`, `DF`).

#### Volume não puxa ticket

`SP` responde por `241` dos `851` pedidos entregues (aproximadamente `28,3%` do volume) e `R$ 182.907,38` em receita, a maior do ranking; mesmo assim seu ticket médio (`R$ 758,95`) fica em `6º`. `SC` inverte o padrão: volume moderado (`59`) e ticket médio no topo (`R$ 1.134,71`). Estratégia por UF baseada apenas em receita total ignora o comportamento de cesta; ticket premium pode existir em mercado menor.

#### Conexão com Q4, região Sul

O PRD (Q4) pede análise de campanha de descontos para `RS`, `SC` e `PR`. Os três estados ocupam posições muito distintas neste ranking:

- `SC`: `R$ 1.134,71` (`1º`)
- `PR`: `R$ 746,94` (`7º`)
- `RS`: `R$ 534,38` (`11º`)

A amplitude interna da Sul é de `2,12×` entre `SC` e `RS`. Tratar a região como bloco único, como o enunciado sugere, ignora que `SC` já opera com ticket premium sem desconto, enquanto `RS` está entre os mais baixos do país. Campanha uniforme corre o risco de canibalizar margem em `SC` sem endereçar a causa do ticket baixo em `RS`.

#### Cruzamento com Q3.a

Produtos do top 5 de receita em Q3.a (`Monitor 32" 4K`, `Impressora Laser`, `Monitor 24"`, `Monitor 27" 144Hz`, `Impressora Jato`) têm preço unitário entre `R$ 989` e `R$ 2.499`. UFs com ticket médio na faixa premium provavelmente concentram pedidos dessas categorias. Confirmar essa hipótese exige breakdown UF × categoria, fora do escopo de Q3.d.

---

#### O que isso significa para o negócio

1. **`SC` é mercado premium com volume saudável.** Combinação de `ticket_medio` líder e `59` pedidos entregues coloca o estado como candidato prioritário a ações de retenção e upsell, não desconto.
2. **`SP` domina receita absoluta, não ticket.** Campanhas de expansão em `SP` têm escala, mas ganho marginal por pedido é menor; volume é a alavanca.
3. **Região Sul não é homogênea.** Recomendação para Q4: segmentar RS/SC/PR por perfil antes de desenhar campanha única.
4. **UFs de amostra muito pequena exigem cautela.** `GO` (`19`), `PA` (`14`), `DF` (`17`) aparecem no extremo do ranking com base que torna o ticket sensível a um ou dois pedidos atípicos.

---

#### Ressalvas

- **`12` pedidos entregues sem UF excluídos:** análise cobre `98,6%` do universo entregue. Nenhum tratamento de imputação foi aplicado, conforme DI-006 em Q2.
- **Amostras pequenas viciam extremos:** bases `< 20` pedidos (`GO`, `PA`, `DF`) produzem ticket volátil. `GO` e `PA` no top 3 podem ser ruído, não padrão estrutural.
- **Sem janela temporal:** o dataset agrega `2024-01-01` a `2024-12-31`. Sazonalidade regional (fim de ano, Black Friday) não aparece.
- **Sem breakdown por categoria:** ticket alto pode vir de mix de produto (ex: concentração de monitores) mais que de comportamento do consumidor da UF. A pergunta respondida é "quanto vale um pedido entregue médio neste estado", não "por que ele tem esse valor".
- **Apenas `status = 'entregue'`:** `em_transito` ficou fora. Estados com proporção maior de pedidos ainda em trânsito aparecem com receita realizada subestimada em relação a estados com pipeline mais antigo.

# Bloco 5: Q4 - Negócio

## Q4 - Analise de Negocio Aberta

_A preencher via `/analisar-negocio`._

# Bloco 6: Q5 - Debug

## Q5 - Encontre o Erro

_A preencher via `/encontrar-erro`._

# Bloco 7: Q6 - Modelagem Dimensional

## Q6 - Modelagem

_A preencher via `/modelar-dimensional` (markdown-only)._

# Bloco 8: Q7 - Insight

## Q7 - Pergunta Livre

_A preencher via `/insight-livre`._